# LAB 03 - Word Representations and Embeddings

In [7]:
!pip install gensim

In [8]:
import time
import numpy as np
import pandas as pd
from gensim.models import Word2Vec

from cooccurrence import build_vocabulary
from cooccurrence import build_cooccurrence_matrix
from cooccurrence import cosine_similarity
from cooccurrence import most_similar

## Corpus

In [9]:
corpus = [
    "the doctor treated the patient",
    "the physician treated the patient",
    "the doctor works in the hospital",
    "the physician works in the hospital",
    "the nurse helped the patient",
    "the nurse works in the hospital",
    "the patient has a disease",
    "the doctor diagnosed the disease",
    "the physician diagnosed the disease",
    "medical treatment helped the patient",
    "clinical treatment helped the patient",
    "the hospital provides medical treatment",
    "the doctor prescribed treatment",
    "the physician prescribed treatment",
    "the nurse provided clinical care",
    "the doctor provided medical care",
    "the physician provided medical care",
    "the patient received treatment",
    "the car moves on the road",
    "the automobile moves on the road",
    "the car has an engine",
    "the automobile has an engine",
    "the driver drives the car",
    "the driver drives the automobile",
    "the cat eats fish",
    "the dog eats meat",
    "the cat likes milk",
    "the dog likes food",
    "the cat is an animal",
    "the dog is an animal",
    "the king rules the kingdom",
    "the queen rules the kingdom",
    "the man is a king",
    "the woman is a queen",
    "the king is a man",
    "the queen is a woman",
    "computer systems process data",
    "computer programs use data",
    "banana is a fruit",
    "banana contains sugar",
    "football is a sport",
    "players play football"
]

tokenized_corpus = [sentence.lower().split() for sentence in corpus]
len(tokenized_corpus)

42

## Experiment 1 - Word-context representation

In [10]:
vocab = build_vocabulary(tokenized_corpus)

rows = []

for window in [1, 2, 5]:
    X = build_cooccurrence_matrix(tokenized_corpus, vocab, window)

    rows.append({
        "window": window,
        "vocabulary_size": len(vocab),
        "matrix_shape": str(X.shape),
        "non_zero": np.count_nonzero(X)
    })

pd.DataFrame(rows)

,window,vocabulary_size,matrix_shape,non_zero
0,1,61,"(61, 61)",196
1,2,61,"(61, 61)",310
2,5,61,"(61, 61)",397


In [11]:
pairs = [
    ("doctor", "physician"),
    ("doctor", "hospital"),
    ("cat", "dog")
]

rows = []

for window in [1, 2, 5]:
    X = build_cooccurrence_matrix(tokenized_corpus, vocab, window)

    for word1, word2 in pairs:
        i = vocab.index(word1)
        j = vocab.index(word2)

        rows.append({
            "window": window,
            "word1": word1,
            "word2": word2,
            "similarity": cosine_similarity(X[i], X[j])
        })

pd.DataFrame(rows)

,window,word1,word2,similarity
0,1,doctor,physician,1.000000
1,1,doctor,hospital,0.885615
2,1,cat,dog,1.000000
3,2,doctor,physician,1.000000
4,2,doctor,hospital,0.815700
5,2,cat,dog,0.866667
6,5,doctor,physician,1.000000
7,5,doctor,hospital,0.859235
8,5,cat,dog,0.875000


In [12]:
X = build_cooccurrence_matrix(tokenized_corpus, vocab, 2)
most_similar("doctor", X, vocab, top_k=5)

[('physician', 1.0),
 ('nurse', 0.896013635174924),
 ('works', 0.8603090020146067),
 ('patient', 0.847801045943832),
 ('driver', 0.8292884288627292)]

### Experiment 1 - Answer

- Vocabulary size không đổi khi tăng window: luôn bằng **61**.
- Matrix size cũng không đổi: luôn là **61 x 61**.
- Số non-zero tăng theo window:
  - `window=1`: **196**
  - `window=2`: **310**
  - `window=5`: **397**

Similarity:

- `doctor-physician`: **1.000 -> 1.000 -> 1.000**
- `doctor-hospital`: **0.886 -> 0.816 -> 0.859**
- `cat-dog`: **1.000 -> 0.867 -> 0.875**

Khi window lớn hơn, mỗi từ có nhiều context hơn nên số non-zero tăng và similarity có thể thay đổi.

Với `window=2`, Top-5 gần `doctor` là:

1. `physician`: **1.000**
2. `nurse`: **0.896**
3. `works`: **0.860**
4. `patient`: **0.848**
5. `driver`: **0.829**

`physician`, `nurse`, `works`, `patient` hợp lý vì chúng xuất hiện trong các context y tế gần `doctor`. `driver` là kết quả nhiễu do corpus nhỏ.

## Experiment 2 - Word2Vec

In [13]:
model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=100,
    window=5,
    min_count=1,
    epochs=100,
    sg=1,
    seed=42,
    workers=1
)

print("Vocabulary size:", len(model.wv))
print("Vector size:", model.vector_size)
print("Window:", model.window)
print("Min count:", model.min_count)
print("Epochs:", model.epochs)
print("Objective: Skip-gram")

Vocabulary size: 61
Vector size: 100
Window: 5
Min count: 1
Epochs: 100
Objective: Skip-gram


In [14]:
words = [
    "doctor",
    "hospital",
    "patient",
    "disease",
    "computer",
    "football",
    "banana"
]

for word in words:
    print("\n", word)
    print(model.wv.most_similar(word, topn=5))


 doctor
[('the', 0.8105359077453613), ('cat', 0.8091229200363159), ('an', 0.8061487674713135), ('queen', 0.8008988499641418), ('is', 0.7951777577400208)]

 hospital
[('the', 0.8986419439315796), ('automobile', 0.8808531761169434), ('programs', 0.87088942527771), ('care', 0.863743007183075), ('is', 0.8614342212677002)]

 patient
[('the', 0.8792431354522705), ('an', 0.8581053614616394), ('data', 0.8536487221717834), ('a', 0.8522868752479553), ('process', 0.8467468619346619)]

 disease
[('computer', 0.7877735495567322), ('is', 0.775935173034668), ('the', 0.77268385887146), ('a', 0.7610087990760803), ('programs', 0.7439013719558716)]

 computer
[('the', 0.919091522693634), ('cat', 0.8965020775794983), ('on', 0.8893544673919678), ('a', 0.887083888053894), ('is', 0.8840857744216919)]

 football
[('the', 0.9299790263175964), ('is', 0.8987195491790771), ('an', 0.8929142951965332), ('cat', 0.8919021487236023), ('a', 0.8892696499824524)]

 banana
[('the', 0.9279802441596985), ('automobile', 0.9

### Experiment 2 - Answer

Model được train với:

- Corpus: **42 câu**
- Vocabulary size: **61**
- Embedding dimension: **100**
- Window: **5**
- Min count: **1**
- Epochs: **100**
- Objective: **Skip-gram**

Top-5 gần `doctor`:

1. `the`: **0.811**
2. `cat`: **0.809**
3. `an`: **0.806**
4. `queen`: **0.801**
5. `is`: **0.795**

Kết quả còn khá nhiễu. `doctor` thường xuất hiện cùng các từ phổ biến như `the`, còn `cat`, `queen`, `an`, `is` nằm trong các mẫu câu ngắn có cấu trúc lặp lại. Vì corpus chỉ có 42 câu và `window=5` khá rộng, model dễ học frequency và pattern câu hơn là semantic similarity thực sự.

## Experiment 3 - Context window

In [15]:
window_models = {}

for window in [2, 5, 10]:
    window_models[window] = Word2Vec(
        sentences=tokenized_corpus,
        vector_size=100,
        window=window,
        min_count=1,
        epochs=100,
        sg=1,
        seed=42,
        workers=1
    )

rows = []

for word1, word2 in pairs:
    row = {
        "word_pair": word1 + " - " + word2
    }

    for window in [2, 5, 10]:
        row["window_" + str(window)] = window_models[window].wv.similarity(word1, word2)

    rows.append(row)

pd.DataFrame(rows)

,word_pair,window_2,window_5,window_10
0,doctor - physician,0.581007,0.705643,0.809687
1,doctor - hospital,0.537007,0.720403,0.850684
2,cat - dog,0.752265,0.862187,0.893674


### Experiment 3 - Answer

Kết quả:

| Word pair | Window 2 | Window 5 | Window 10 |
|---|---:|---:|---:|
| doctor-physician | 0.581 | 0.706 | 0.810 |
| doctor-hospital | 0.537 | 0.720 | 0.851 |
| cat-dog | 0.752 | 0.862 | 0.894 |

Trong lần chạy này, similarity của cả ba cặp đều tăng khi window tăng.

**Window lớn hơn có luôn tốt hơn không?**

Không. Window lớn lấy được context rộng hơn nên có thể học tốt quan hệ chủ đề, nhưng cũng đưa thêm context không liên quan và làm mất các quan hệ local. Vì vậy kết quả phụ thuộc corpus và task.

## Experiment 4 - Embedding dimension

In [16]:
dimension_models = {}
dimension_results = []

documents = [
    "the doctor treated the patient",
    "the hospital provides medical treatment",
    "the automobile has an engine",
    "computer programs use data",
    "the patient received clinical care",
    "players play football"
]

def sentence_vector(sentence, model):
    vectors = []

    for word in sentence.lower().split():
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

for dimension in [50, 100, 300]:
    start = time.time()

    m = Word2Vec(
        sentences=tokenized_corpus,
        vector_size=dimension,
        window=5,
        min_count=1,
        epochs=100,
        sg=1,
        seed=42,
        workers=1
    )

    training_time = time.time() - start
    model_size = m.wv.vectors.nbytes

    analogy = m.wv.most_similar(
        positive=["king", "woman"],
        negative=["man"],
        topn=1
    )[0]

    query_vector = sentence_vector("medical treatment", m)
    search_scores = []

    for document in documents:
        document_vector = sentence_vector(document, m)
        score = cosine_similarity(query_vector, document_vector)
        search_scores.append((document, score))

    search_scores.sort(key=lambda x: x[1], reverse=True)

    dimension_models[dimension] = m

    dimension_results.append({
        "dimension": dimension,
        "training_time": training_time,
        "model_size_bytes": model_size,
        "doctor_physician": m.wv.similarity("doctor", "physician"),
        "doctor_hospital": m.wv.similarity("doctor", "hospital"),
        "cat_dog": m.wv.similarity("cat", "dog"),
        "analogy_top1": analogy[0],
        "analogy_score": analogy[1],
        "search_top1": search_scores[0][0],
        "search_score": search_scores[0][1]
    })

pd.DataFrame(dimension_results)

,dimension,training_time,model_size_bytes,doctor_physician,doctor_hospital,cat_dog,analogy_top1,analogy_score,search_top1,search_score
0,50,0.112354,12200,0.803190,0.689268,0.814493,hospital,0.642137,the hospital provides medical treatment,0.970638
1,100,0.116751,24400,0.705643,0.720403,0.862187,banana,0.481243,the hospital provides medical treatment,0.965989
2,300,0.130713,73200,0.701582,0.723084,0.862346,clinical,0.473933,the hospital provides medical treatment,0.970937


### Experiment 4 - Answer

So sánh `50`, `100`, `300` dimensions:

| Dimension | Model size | doctor-physician | doctor-hospital | cat-dog | Analogy Top-1 | Semantic search Top-1 |
|---|---:|---:|---:|---:|---|---|
| 50 | 12,200 bytes | 0.803 | 0.689 | 0.814 | `hospital` (0.642) | `the hospital provides medical treatment` (0.971) |
| 100 | 24,400 bytes | 0.706 | 0.720 | 0.862 | `banana` (0.481) | `the hospital provides medical treatment` (0.966) |
| 300 | 73,200 bytes | 0.702 | 0.723 | 0.862 | `clinical` (0.474) | `the hospital provides medical treatment` (0.971) |

Training time giữa ba model chỉ chênh lệch rất nhỏ vì corpus nhỏ, nên chưa thấy ảnh hưởng rõ ràng.

Model size tăng gần tuyến tính khi dimension tăng.

Similarity không tăng đều. `doctor-physician` thậm chí giảm khi tăng dimension, còn `doctor-hospital` và `cat-dog` chỉ tăng nhẹ.

Analogy `king - man + woman` không trả về `queen` ở cả ba dimension, nên tăng dimension không làm analogy tốt hơn trong corpus này.

Downstream semantic search cho cùng Top-1 document ở cả ba dimension và score gần như tương đương, khoảng `0.966-0.971`.

**Kết luận:** tăng dimension làm model lớn hơn và tăng capacity, nhưng với corpus nhỏ này không đảm bảo similarity, analogy hay downstream task tốt hơn.

## Evaluation - Word similarity

In [17]:
evaluation_pairs = [
    ("doctor", "physician"),
    ("car", "automobile"),
    ("king", "queen"),
    ("cat", "dog"),
    ("computer", "banana")
]

similarity_results = []

for word1, word2 in evaluation_pairs:
    similarity_results.append({
        "word1": word1,
        "word2": word2,
        "similarity": model.wv.similarity(word1, word2)
    })

similarity_df = pd.DataFrame(similarity_results)
similarity_df = similarity_df.sort_values("similarity", ascending=False)
similarity_df

,word1,word2,similarity
1,car,automobile,0.872367
3,cat,dog,0.862187
4,computer,banana,0.855171
0,doctor,physician,0.705643
2,king,queen,0.671919


Analysis:

## Evaluation - Word analogy

In [18]:
model.wv.most_similar(
    positive=["king", "woman"],
    negative=["man"],
    topn=5
)

[('banana', 0.4812430441379547),
 ('rules', 0.4705253839492798),
 ('dog', 0.46037283539772034),
 ('automobile', 0.45415621995925903),
 ('animal', 0.44729623198509216)]

Analysis:

## Application - Semantic search

In [19]:
documents = [
    "the doctor treated the patient",
    "the hospital provides medical treatment",
    "the automobile has an engine",
    "computer programs use data",
    "the patient received clinical care",
    "players play football"
]

def sentence_vector(sentence, model):
    words = sentence.lower().split()
    vectors = []

    for word in words:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)


def semantic_search(query, documents, model, top_k=3):
    query_vector = sentence_vector(query, model)
    scores = []

    for document in documents:
        document_vector = sentence_vector(document, model)
        score = cosine_similarity(query_vector, document_vector)
        scores.append((document, score))

    scores.sort(key=lambda x: x[1], reverse=True)

    return scores[:top_k]


semantic_search("medical treatment", documents, model, top_k=3)

[('the hospital provides medical treatment', 0.965989179725363),
 ('the doctor treated the patient', 0.9439054746003357),
 ('the patient received clinical care', 0.9436430116153921)]

### Application - Answer

Query: `medical treatment`

Top-3 documents:

1. `the hospital provides medical treatment` — **0.966**
2. `the doctor treated the patient` — **0.944**
3. `the patient received clinical care` — **0.944**

Các kết quả đều liên quan đến chủ đề y tế. Semantic search có thể tìm được các document không chứa đúng toàn bộ từ trong query vì word embeddings biểu diễn quan hệ ngữ nghĩa giữa các từ như `doctor`, `patient`, `clinical`, `care` và `treatment`.

So với TF-IDF, semantic search không chỉ dựa vào từ trùng khớp trực tiếp mà còn dựa vào độ gần của các vector trong embedding space.

## 29. Individual learning check

CBOW khác Skip-gram ở đâu?

khác ở CBOW học từ target từ context ví dụ là tôi ____ học đoán từ đi còn skip-gram thì ngược lại ví dụ cho từ đi và đoán context xung quanh là tôi và học

## Results export

In [20]:
similarity_df.to_csv("results.csv", index=False)
print("Saved results.csv")

Saved results.csv
